# Tideway Admin API Examples

This notebook shows how to initialise a Tideway appliance client and call common Admin endpoints using the top-level REST wrappers. Replace the placeholder values before running.

## Setup
- Ensure `tideway` is installed in your environment (e.g. `pip install -e .` from the repo root).
- Provide your appliance hostname and API token below. Do **not** commit tokens.

In [ ]:
import tideway

# Configuration
TARGET = '' # e.g. 'discovery.example.com'
TOKEN = '' # keep secrets out of source control
API_VERSION = '1.16'                # latest supported API
SSL_VERIFY = False                  # set to True when using valid certs

tw = tideway.appliance(TARGET, TOKEN, api_version=API_VERSION, ssl_verify=SSL_VERIFY)

def show_response(resp, label):
    if resp.ok:
        try:
            return resp.json()
        except Exception:
            return resp.text
    try:
        body = resp.json()
    except Exception:
        body = resp.text
    return {'endpoint': label, 'status': resp.status_code, 'body': body}

tw.api_about.json()  # quick sanity check


## Admin calls
Examples of common admin endpoints.

In [ ]:
# Appliance baseline summary (appliance only - won't work on Helix)
admin_baseline = tw.get('/admin/baseline')
show_response(admin_baseline, '/admin/baseline')


In [ ]:
# Appliance details
admin_about = tw.get('/admin/about')
show_response(admin_about, '/admin/about')


In [ ]:
# Licensing endpoints with explicit response types
try:
    admin_licensing_text = tw.get('/admin/licensing', response='text/plain')
    admin_licensing_csv = tw.get('/admin/licensing/csv', response='application/zip')
    admin_licensing_raw = tw.get('/admin/licensing/raw', response='application/zip')
except TypeError:
    admin_licensing_text = tw.get('/admin/licensing')
    admin_licensing_csv = tw.get('/admin/licensing/csv')
    admin_licensing_raw = tw.get('/admin/licensing/raw')

# Render plain text with real line breaks when available
if admin_licensing_text.ok:
    print(admin_licensing_text.text)
else:
    show_response(admin_licensing_text, '/admin/licensing')


In [ ]:
# Fetch API schema (swagger/openapi)
api_swagger = tw.api_swagger
api_swagger.status_code, api_swagger.ok

if api_swagger.ok:
    schema = api_swagger.json()
    list(schema.keys())[:10]
else:
    api_swagger.text


In [ ]:
# Parsed schema via helper
parsed_schema = tw.api_schema()
list(parsed_schema.keys())[:10]


In [ ]:
# Inspect available API paths
paths = tw.api_paths()
list(paths.keys())[:10]


In [ ]:
# Endpoint help (prints to stdout)
tw.api_help


## Direct admin GET calls
Raw examples using the generic `get` wrapper without helper methods. Adjust `response` headers as needed.

In [ ]:
# Direct admin endpoints
admin_instance = tw.get('/admin/instance')
admin_cluster = tw.get('/admin/cluster')
admin_organizations = tw.get('/admin/organizations')
admin_preferences = tw.get('/admin/preferences')
admin_builtin_reports = tw.get('/admin/builtin_reports')
admin_custom_reports = tw.get('/admin/custom_reports')
admin_smtp = tw.get('/admin/smtp')

# Collect all responses with context
admin_results = {
    '/admin/instance': show_response(admin_instance, '/admin/instance'),
    '/admin/cluster': show_response(admin_cluster, '/admin/cluster'),
    '/admin/organizations': show_response(admin_organizations, '/admin/organizations'),
    '/admin/preferences': show_response(admin_preferences, '/admin/preferences'),
    '/admin/builtin_reports': show_response(admin_builtin_reports, '/admin/builtin_reports'),
    '/admin/custom_reports': show_response(admin_custom_reports, '/admin/custom_reports'),
    '/admin/smtp': show_response(admin_smtp, '/admin/smtp'),
}
admin_results
